# Cross-Validation and Hyperparameter Tuning

Techniques for robust model evaluation and optimization. Cross-validation gives a more reliable estimate of model performance than a single train/test split. Hyperparameter tuning finds the best configuration for your model.

## Why Cross-Validation?

A single train/test split has high variance — the performance metric depends heavily on *which* samples end up in the test set. Cross-validation reduces this by averaging performance across multiple splits.

### k-Fold Cross-Validation

1. Split data into *k* equal folds
2. For each fold: train on *k-1* folds, evaluate on the held-out fold
3. Report the average score across all *k* folds

### Stratified k-Fold

Preserves the class proportion in each fold — crucial for imbalanced datasets.

### Typical Choices
- **k=5** or **k=10**: good balance of bias and variance
- **k=n** (Leave-One-Out): computationally expensive, low bias

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_digits, load_diabetes
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate, KFold, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV, validation_curve, learning_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, make_scorer

sns.set_style('whitegrid')
print("Libraries imported.")

In [ ]:
# Load dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Classes: malignant={np.sum(y==0)}, benign={np.sum(y==1)}")

In [ ]:
# Single train/test split vs Cross-validation
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

svm = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)
svm.fit(X_train, y_train)
single_score = accuracy_score(y_test, svm.predict(X_test))
print(f"Single train/test split accuracy: {single_score:.4f}")

# 5-fold cross-validation
cv_scores = cross_val_score(svm, X_scaled, y, cv=5, scoring='accuracy')
print(f"5-fold CV scores: {cv_scores.round(4)}")
print(f"CV mean: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

In [ ]:
# StratifiedKFold vs KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("KFold — class distribution per fold:")
for i, (train_idx, val_idx) in enumerate(kf.split(X_scaled, y)):
    y_train_fold = y[train_idx]
    print(f"  Fold {i}: class 0 = {np.sum(y_train_fold==0)}, class 1 = {np.sum(y_train_fold==1)}")

print("\nStratifiedKFold — class distribution per fold:")
for i, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
    y_train_fold = y[train_idx]
    print(f"  Fold {i}: class 0 = {np.sum(y_train_fold==0)}, class 1 = {np.sum(y_train_fold==1)}")

In [ ]:
# cross_validate — get multiple metrics
scoring = ['accuracy', 'precision', 'recall', 'f1']
cv_results = cross_validate(svm, X_scaled, y, cv=5, scoring=scoring, return_train_score=True)

print("Cross-validate results:")
for metric in scoring:
    test_scores = cv_results[f'test_{metric}']
    train_scores = cv_results[f'train_{metric}']
    print(f"  {metric:10s}  train: {train_scores.mean():.4f}  test: {test_scores.mean():.4f}")

## Hyperparameter Tuning

### GridSearchCV
- Exhaustively searches all combinations in a parameter grid
- Guarantees finding the best combination (within the grid)
- Computationally expensive for large grids

### RandomizedSearchCV
- Samples random combinations from parameter distributions
- More efficient for high-dimensional spaces
- Can explore more values with fewer iterations

In [ ]:
# GridSearchCV — SVM hyperparameter tuning
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': [0.01, 0.1, 1],
    'kernel': ['rbf']
}

grid_search = GridSearchCV(
    SVC(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_scaled, y)

print("GridSearchCV Results:")
print(f"  Best parameters: {grid_search.best_params_}")
print(f"  Best CV score: {grid_search.best_score_:.4f}")
print(f"\nAll results:")
results_df = pd.DataFrame(grid_search.cv_results_)
print(results_df[['params', 'mean_test_score', 'std_test_score']].to_string())

In [ ]:
# RandomizedSearchCV — more efficient exploration
from scipy.stats import expon, randint

param_dist = {
    'n_estimators': randint(10, 200),
    'max_depth': [3, 5, 10, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10)
}

rf_random = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), param_dist,
    n_iter=30, cv=5, scoring='accuracy', random_state=42, n_jobs=-1
)
rf_random.fit(X_scaled, y)

print("RandomizedSearchCV for Random Forest:")
print(f"  Best parameters: {rf_random.best_params_}")
print(f"  Best CV score: {rf_random.best_score_:.4f}")

In [ ]:
# Validation Curve — effect of a single parameter
train_scores, test_scores = validation_curve(
    SVC(kernel='rbf', random_state=42), X_scaled, y,
    param_name='C', param_range=[0.001, 0.01, 0.1, 1, 10, 100],
    cv=5, scoring='accuracy'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.semilogx([0.001, 0.01, 0.1, 1, 10, 100], train_mean, 'o-', label='Training score', color='blue')
plt.fill_between([0.001, 0.01, 0.1, 1, 10, 100], train_mean - train_std, train_mean + train_std, alpha=0.2, color='blue')
plt.semilogx([0.001, 0.01, 0.1, 1, 10, 100], test_mean, 'o-', label='Cross-validation score', color='red')
plt.fill_between([0.001, 0.01, 0.1, 1, 10, 100], test_mean - test_std, test_mean + test_std, alpha=0.2, color='red')
plt.xlabel('C (regularization parameter)')
plt.ylabel('Accuracy')
plt.title('Validation Curve: Effect of C on SVM Performance')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Learning Curve — effect of training set size
train_sizes, train_scores_lc, test_scores_lc = learning_curve(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_scaled, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy', n_jobs=-1
)

train_mean_lc = np.mean(train_scores_lc, axis=1)
test_mean_lc = np.mean(test_scores_lc, axis=1)
train_std_lc = np.std(train_scores_lc, axis=1)
test_std_lc = np.std(test_scores_lc, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean_lc, 'o-', label='Training score', color='blue')
plt.fill_between(train_sizes, train_mean_lc - train_std_lc, train_mean_lc + train_std_lc, alpha=0.2, color='blue')
plt.plot(train_sizes, test_mean_lc, 'o-', label='Cross-validation score', color='red')
plt.fill_between(train_sizes, test_mean_lc - test_std_lc, test_mean_lc + test_std_lc, alpha=0.2, color='red')
plt.xlabel('Training examples')
plt.ylabel('Accuracy')
plt.title('Learning Curve: Performance vs Training Size')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.show()

print("Narrow gap = low variance; convergence = enough data")

In [ ]:
# Detecting overfitting with CV
from sklearn.tree import DecisionTreeClassifier

for depth in [1, 5, 20]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    cv = cross_validate(dt, X_scaled, y, cv=5, return_train_score=True)
    print(f"max_depth={depth:2d}:  train_acc={cv['train_score'].mean():.4f},  "
          f"test_acc={cv['test_score'].mean():.4f},  "
          f"gap={cv['train_score'].mean() - cv['test_score'].mean():.4f}")

In [ ]:
# Nested Cross-Validation — unbiased evaluation of the model family
from sklearn.model_selection import cross_val_score

# Inner loop: find best params
# Outer loop: estimate performance

param_grid_small = {'C': [0.1, 1, 10], 'gamma': [0.01, 0.1]}

svm_outer = SVC(kernel='rbf', random_state=42)

# Manual nested CV
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
outer_scores = []

for train_idx, test_idx in outer_cv.split(X_scaled, y):
    X_outer_train, X_outer_test = X_scaled[train_idx], X_scaled[test_idx]
    y_outer_train, y_outer_test = y[train_idx], y[test_idx]
    
    # Inner CV
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    gs = GridSearchCV(svm_outer, param_grid_small, cv=inner_cv, scoring='accuracy')
    gs.fit(X_outer_train, y_outer_train)
    
    # Evaluate on outer test fold
    score = accuracy_score(y_outer_test, gs.predict(X_outer_test))
    outer_scores.append(score)

print(f"Nested CV accuracy: {np.mean(outer_scores):.4f} +/- {np.std(outer_scores):.4f}")
print("Note: This gives an unbiased estimate of the model's performance.")

In [ ]:
# Problem: Tune Random Forest for best parameters on digits
digits = load_digits()
X_d, y_d = digits.data, digits.target

param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42), param_grid_rf, cv=3, scoring='accuracy', n_jobs=-1
)
grid_rf.fit(X_d, y_d)

print("Random Forest Tuning on Digits:")
print(f"  Best params: {grid_rf.best_params_}")
print(f"  Best CV accuracy: {grid_rf.best_score_:.4f}")
print(f"  Test accuracy: {accuracy_score(y_d, grid_rf.predict(X_d)):.4f}")

In [ ]:
# Compare CV strategies: KFold vs StratifiedKFold
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)

kf_scores = cross_val_score(lr, X_scaled, y, cv=KFold(5, shuffle=True, random_state=42))
skf_scores = cross_val_score(lr, X_scaled, y, cv=StratifiedKFold(5, shuffle=True, random_state=42))

print(f"KFold CV:             {kf_scores.mean():.4f} +/- {kf_scores.std():.4f}")
print(f"StratifiedKFold CV:   {skf_scores.mean():.4f} +/- {skf_scores.std():.4f}")
print("\nStratifiedKFold is preferred for classification (preserves class proportions).")

In [ ]:
# Interview Problem: When to use GridSearch vs RandomizedSearch?
print("""
Interview Question: Compare GridSearchCV and RandomizedSearchCV.

GridSearchCV:
- Exhaustive search over all specified parameter combinations
- Guarantees finding the best combination IN the grid
- Time increases exponentially with number of parameters
- Best for small parameter spaces (<= 3 parameters, few values each)

RandomizedSearchCV:
- Samples random combinations from specified distributions
- Does NOT guarantee finding the absolute best
- Can cover more values per parameter with same budget
- Best for large parameter spaces (>= 4 parameters)
- Often finds near-optimal solutions much faster
""")

In [ ]:
# Summary
print("=" * 60)
print("CROSS-VALIDATION & HYPERPARAMETER TUNING — KEY TAKEAWAYS")
print("=" * 60)
print("""
1. Cross-validation gives more reliable performance estimates than single split
2. StratifiedKFold preserves class proportions — use for classification
3. cross_val_score for quick CV, cross_validate for detailed metrics
4. GridSearchCV: exhaustive but expensive
5. RandomizedSearchCV: efficient for high-dimensional searches
6. Validation curve: see effect of one parameter
7. Learning curve: diagnose bias/variance with training set size
8. Nested CV: unbiased estimate of model family performance
""")